In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('AnimeRecommender') \
    .config('spark.ui.port', '4040') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print('Spark UI: http://localhost:4040')

Spark version: 3.5.0
Spark UI: http://localhost:4040


In [2]:
BASE = '/home/jovyan/work/data/preprocessed'

reviews_data = spark.read.csv(f'{BASE}/reviewsV2.csv', header=True, inferSchema=True)
anime_data   = spark.read.csv(f'{BASE}/animes.csv',    header=True, inferSchema=True)

print('reviews schema:')
reviews_data.printSchema()
print('animes schema:')
anime_data.printSchema()

reviews schema:
root
 |-- _c0: integer (nullable = true)
 |-- uid: integer (nullable = true)
 |-- profile: string (nullable = true)
 |-- anime_uid: integer (nullable = true)
 |-- score: integer (nullable = true)

animes schema:
root
 |-- _c0: integer (nullable = true)
 |-- anime_id: integer (nullable = true)
 |-- genre: string (nullable = true)



In [3]:
from pyspark.ml.feature import StringIndexer
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col('user_id').cast('integer'),
    col('anime_uid').cast('integer'),
    col('score').cast('float')
).filter(col('score') > 0).filter( col('score') <= 10)  # drop unrated rows

print(f'Total rated reviews: {final_data.count():,}')
final_data.show(7)

Total rated reviews: 130,517
+-------+---------+-----+
|user_id|anime_uid|score|
+-------+---------+-----+
|     32|    34096|  8.0|
|   1104|    34599| 10.0|
|   1825|    28891|  7.0|
|   3796|     2904|  9.0|
|   9589|     4181| 10.0|
|   9872|     2904| 10.0|
|    554|    16664|  6.0|
+-------+---------+-----+
only showing top 7 rows



In [4]:
anime_data.select('anime_id', 'genre').show(5, truncate=False)

+--------+-------------------------------------------------------------------------------------+
|anime_id|genre                                                                                |
+--------+-------------------------------------------------------------------------------------+
|28891   |['Comedy', 'Sports', 'Drama', 'School', 'Shounen']                                   |
|23273   |['Drama', 'Music', 'Romance', 'School', 'Shounen']                                   |
|34599   |['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'Fantasy']                               |
|5114    |['Action', 'Military', 'Adventure', 'Comedy', 'Drama', 'Magic', 'Fantasy', 'Shounen']|
|31758   |['Action', 'Mystery', 'Supernatural', 'Vampire']                                     |
+--------+-------------------------------------------------------------------------------------+
only showing top 5 rows



In [5]:
from pyspark.ml.feature import CountVectorizer, IDF, Tokenizer
from pyspark.sql.functions import split, lower, trim, col, regexp_replace

# Clean and split the genre string into an array of tokens
anime_clean = anime_data \
    .withColumnRenamed('anime_id', 'anime_uid') \
    .withColumn('genre_stripped',
        regexp_replace(col('genre'), r"[\[\]']", '')  # remove [ ] and '
    ) \
    .withColumn('genres_array',
        split(lower(trim(col('genre_stripped'))), ',\\s*')
    ) \
    .drop('genre_stripped') \
    .dropna(subset=['genre'])

anime_clean.select('anime_uid', 'genre', 'genres_array').show(5, truncate=False)

+---------+-------------------------------------------------------------------------------------+---------------------------------------------------------------------+
|anime_uid|genre                                                                                |genres_array                                                         |
+---------+-------------------------------------------------------------------------------------+---------------------------------------------------------------------+
|28891    |['Comedy', 'Sports', 'Drama', 'School', 'Shounen']                                   |[comedy, sports, drama, school, shounen]                             |
|23273    |['Drama', 'Music', 'Romance', 'School', 'Shounen']                                   |[drama, music, romance, school, shounen]                             |
|34599    |['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'Fantasy']                               |[sci-fi, adventure, mystery, drama, fantasy]                   

In [6]:
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.ml import Pipeline

cv  = CountVectorizer(inputCol='genres_array', outputCol='tf')
idf = IDF(inputCol='tf', outputCol='tfidf')

pipeline = Pipeline(stages=[cv, idf])
anime_vectorized = pipeline.fit(anime_clean).transform(anime_clean)

anime_vectorized.select('anime_uid', 'tfidf').show(5, truncate=False)

+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|anime_uid|tfidf                                                                                                                                                                           |
+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|28891    |(44,[0,6,7,11,19],[1.0559987199793983,1.8867120635933943,2.1358203951885684,2.3368971113633537,3.171722555549222])                                                              |
|23273    |(44,[6,7,8,9,11],[1.8867120635933943,2.1358203951885684,2.163335358473838,2.219043171321504,2.3368971113633537])                                                                |
|34599    |(44,[2,3,4,6,20],[1.6968249478817246,1.77300

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Pull vectors to driver — fine for a few thousand anime
anime_pd = anime_vectorized.select('anime_uid', 'tfidf').toPandas()
anime_pd['vec'] = anime_pd['tfidf'].apply(lambda v: v.toArray())

# Build matrix and compute all pairwise similarities at once
matrix = np.vstack(anime_pd['vec'].values)
sim_matrix = cosine_similarity(matrix)  # shape: (n_anime, n_anime)

# Map index → anime_uid
idx_to_uid = anime_pd['anime_uid'].to_dict()
uid_to_idx = {v: k for k, v in idx_to_uid.items()}

In [8]:
def content_recommend(anime_uid, n=10):
    if anime_uid not in uid_to_idx:
        print(f'anime_uid {anime_uid} not found')
        return

    idx = uid_to_idx[anime_uid]
    scores = sim_matrix[idx]

    # Get top-n most similar (excluding itself)
    top_indices = np.argsort(scores)[::-1][1:n+1]
    
    results = pd.DataFrame({
        'anime_uid':  [idx_to_uid[i] for i in top_indices],
        'similarity': [scores[i]     for i in top_indices]
    })
    
    # Join genre info back
    genre_map = anime_pd.set_index('anime_uid')['tfidf']  # reuse anime_pd
    genre_lookup = anime_clean.select('anime_uid', 'genre').toPandas() \
                              .set_index('anime_uid')
    results['genre'] = results['anime_uid'].map(genre_lookup['genre'])
    
    return results

# Test with first anime in dataset
sample_uid = anime_pd['anime_uid'].iloc[0]
print(f'Recommendations similar to anime {sample_uid}:')
content_recommend(sample_uid)

Recommendations similar to anime 28891:


,anime_uid,similarity,genre
0,29755,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
1,38883,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
2,40776,1.0,"['Comedy', 'Drama', 'School', 'Shounen', 'Spor..."
3,9077,1.0,"['Sports', 'Comedy', 'School', 'Drama', 'Shoun..."
4,16514,1.0,"['Comedy', 'Drama', 'School', 'Shounen', 'Spor..."
5,35110,1.0,"['Comedy', 'Drama', 'School', 'Shounen', 'Spor..."
6,35111,1.0,"['Comedy', 'Drama', 'School', 'Shounen', 'Spor..."
7,20583,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
8,170,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."
9,40262,1.0,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun..."


In [12]:
from pyspark.ml.recommendation import ALSModel
model = ALSModel.load('/home/jovyan/work/models/user-item')

In [18]:
def hybrid_recommend_v2(user_id, n=10, als_weight=0.5, content_weight=0.5):
    # --- ALS user-item score ---
    target_user = spark.createDataFrame([(user_id,)], ['user_id'])
    als_recs = model.recommendForUserSubset(target_user, 50) \
        .select('user_id', explode('recommendations').alias('rec')) \
        .select('user_id', col('rec.anime_uid'), col('rec.rating').alias('als_score')) \
        .toPandas()

    if als_recs.empty:
        print('No ALS recommendations for this user')
        return

    # --- Content similarity to user's rated anime ---
    user_history = final_data.filter(col('user_id') == user_id) \
        .select('anime_uid').toPandas()['anime_uid'].tolist()

    def avg_content_sim(candidate_uid):
        if candidate_uid not in uid_to_idx:
            return 0.0
        c_idx = uid_to_idx[candidate_uid]
        sims = [
            sim_matrix[uid_to_idx[h]][c_idx]
            for h in user_history if h in uid_to_idx
        ]
        return float(np.mean(sims)) if sims else 0.0

    als_recs['content_score'] = als_recs['anime_uid'].map(avg_content_sim)

    # --- Normalize + blend ---
    for col_name in ['als_score', 'content_score']:
        mn, mx = als_recs[col_name].min(), als_recs[col_name].max()
        als_recs[f'{col_name}_norm'] = (als_recs[col_name] - mn) / (mx - mn + 1e-9)

    als_recs['hybrid_score'] = (
        als_weight    * als_recs['als_score_norm'] +
        content_weight * als_recs['content_score_norm']
    )

    result = als_recs.sort_values('hybrid_score', ascending=False).head(n)

    # Join genre labels
    genre_lookup = anime_clean.select('anime_uid', 'genre').toPandas() \
                              .set_index('anime_uid')
    result['genre'] = result['anime_uid'].map(genre_lookup['genre'])

    return result[['anime_uid', 'genre', 'als_score', 'content_score', 'hybrid_score']]

# Test
target_uid = final_data.select('user_id').first()[0]
hybrid_recommend_v2(target_uid)

,anime_uid,genre,als_score,content_score,hybrid_score
0,9806,"['Action', 'Historical', 'Martial Arts', 'Fant...",14.859090,0.120096,0.717712
1,823,"['Hentai', 'Romance']",14.152180,0.135514,0.660672
2,6266,"['Drama', 'School', 'Slice of Life']",12.762347,0.222322,0.650947
22,704,"['Drama', 'Romance', 'Slice of Life', 'Superna...",11.213549,0.275814,0.561712
24,10030,"['Comedy', 'Drama', 'Romance', 'Shounen']",11.197163,0.268693,0.546833
11,1145,"['Action', 'Mystery', 'Drama', 'Shounen']",11.938839,0.177139,0.470030
7,4051,"['Action', 'Adventure', 'Comedy', 'Kids', 'Sci...",12.110229,0.165676,0.469855
10,168,"['Action', 'Sci-Fi', 'Adventure', 'Super Power...",11.955916,0.174412,0.467139
13,38234,"['Action', 'Adventure', 'Comedy', 'Super Power...",11.503368,0.203495,0.465455
5,9969,"['Action', 'Sci-Fi', 'Comedy', 'Historical', '...",12.560355,0.125769,0.451628
